In [ ]:
"""
CAFA5 PPI Protein Name Cleaning Pipeline
=========================================

This notebook cleans and standardizes protein names in the CAFA5 PPI (Protein-Protein
Interaction) metadata dataset to optimize for Large Language Model (LLM) training.

Cleaning Operations:
- Remove EC classification numbers (database annotations)
- Remove cellular location/compartment suffixes
- Remove cofactor specifications in brackets
- Remove complex protein descriptions (Includes, Cleaved into)
- Remove redundant long alternative names
- Preserve chemical stereochemistry notation
- Preserve essential functional domain information

Input:
- CAFA5 PPI metadata from HuggingFace (wanglab/cafa5, ppi_metadata)

Output:
- Cleaned dataset in Parquet format
- CSV backup for compatibility
- Comprehensive analysis visualizations
- Statistical summary reports

Impact:
- Reduces protein name length by average of 30-40%
- Maintains biological accuracy and protein identity
- Optimizes for LLM training by removing database-specific metadata

Author

NOTE: Clone repository first. Dataset will be downloaded from HuggingFace automatically.
"""

In [ ]:
"""
Install Required Packages
-------------------------
Install necessary Python packages for dataset processing and analysis.
"""

import sys

print("Installing required packages...")
!pip install -q datasets matplotlib seaborn

print("Package installation complete")

In [ ]:
"""
Import Libraries and Configure Environment
------------------------------------------
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import re
import os
import numpy as np
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("Libraries imported successfully")

In [ ]:
"""
Configuration Parameters
------------------------
Set output directory for processed files.
Modify OUTPUT_DIR to specify where cleaned datasets will be saved.
"""

# Output directory configuration
OUTPUT_DIR = "data"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define target column for cleaning
PROTEIN_NAME_COLUMN = "Protein names"

print(f"Configuration:")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Target column: {PROTEIN_NAME_COLUMN}")

In [ ]:
"""
Load CAFA5 PPI Metadata Dataset
-------------------------------
Download dataset from HuggingFace repository.
"""

print("Loading CAFA5 PPI metadata dataset from HuggingFace...")
print("This may take a few minutes for first-time download...")

# Load dataset from HuggingFace
dataset = load_dataset("wanglab/cafa5", "ppi_metadata")
df = dataset['metadata'].to_pandas()

print(f"\nDataset loaded successfully")
print(f"  Shape: {df.shape}")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print(f"\nColumns in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

print(f"\nSample of original protein names (first 10):")
original_sample = df[PROTEIN_NAME_COLUMN].dropna().head(10)
for i, name in enumerate(original_sample, 1):
    print(f"  {i:2d}. {name}")

In [ ]:
"""
Protein Name Cleaning Function
------------------------------
Comprehensive cleaning function optimized for LLM training.
"""

def clean_protein_name(name):
    """
    Clean protein name by removing database annotations and metadata.

    Removes:
    - Complex protein descriptions (Includes, Cleaved into)
    - Cellular location/compartment suffixes
    - EC classification numbers
    - Cofactor specifications
    - Long alternative names

    Preserves:
    - Chemical stereochemistry notation
    - Essential functional information
    - Core protein identity

    Args:
        name (str): Original protein name

    Returns:
        str: Cleaned protein name
    """
    if pd.isna(name):
        return name

    name_str = str(name).strip()

    # Remove complex protein descriptions
    name_str = re.sub(r'\s*\[Includes:[^\]]*\]', '', name_str)
    name_str = re.sub(r'\s*\[Cleaved into:[^\]]*\]', '', name_str)

    # Remove cellular location and compartment suffixes
    location_pattern = (
        r',\s*(mitochondrial|chloroplastic|cytoplasmic|cytosolic|nuclear|'
        r'peroxisomal|endoplasmic\s+reticulum|plasma\s+membrane|cell\s+wall|'
        r'vacuolar|ribosomal|membrane|extracellular|lysosomal|golgi)\.?$'
    )
    name_str = re.sub(location_pattern, '', name_str, flags=re.IGNORECASE)

    # Remove EC classification numbers
    name_str = re.sub(r'\s*\(EC\s+[\d\.\-n]+\)', '', name_str)

    # Remove cofactor specifications
    cofactors = [
        'ubiquinone', 'NADP', 'NAD', 'NADH', 'NADPH', 'FAD', 'FMN',
        'ATP', 'ADP', 'GTP', 'GDP', 'Mn', 'Cu-Zn', 'Fe', 'Zn', 'Mg',
        'Ca', 'Mo', 'Ni', 'Co', 'ADP-forming', 'UDP-forming',
        'ADP/GDP-forming', 'GTP-forming', 'glutamine-hydrolyzing',
        'isomerizing', 'ammonia'
    ]

    for cofactor in cofactors:
        pattern = rf'\s*[\[\(]{re.escape(cofactor)}[\]\)]'
        name_str = re.sub(pattern, '', name_str, flags=re.IGNORECASE)

    # Handle alternative names while preserving chemical stereochemistry
    has_chemical_start = re.match(r'^\([SREZ3456789]\w*\)-', name_str)

    if has_chemical_start:
        # For chemical names, only remove very long descriptions
        name_str = re.sub(r'\s*\([^)]{40,}\)', '', name_str)
    else:
        # For standard names, remove long alternatives and apply standard cleaning
        name_str = re.sub(r'\s*\([^)]{30,}\)', '', name_str)
        name_str = name_str.split('(')[0].strip()

    # Standardize whitespace and remove trailing punctuation
    name_str = re.sub(r'\s+', ' ', name_str)
    name_str = re.sub(r'[,\s]+$', '', name_str)

    return name_str.strip()

print("Protein name cleaning function defined")

In [ ]:
"""
Apply Cleaning Function to Dataset
----------------------------------
Process all protein names through cleaning pipeline.
"""

print("Applying protein name cleaning algorithm...")
print(f"Processing {len(df):,} rows - this may take several minutes...")

# Store original names for comparison
original_names = df[PROTEIN_NAME_COLUMN].copy()

# Apply cleaning function
df[PROTEIN_NAME_COLUMN] = df[PROTEIN_NAME_COLUMN].apply(clean_protein_name)

print("Protein name cleaning completed successfully")

print(f"\nSample of cleaned protein names (first 10):")
cleaned_sample = df[PROTEIN_NAME_COLUMN].dropna().head(10)
for i, name in enumerate(cleaned_sample, 1):
    print(f"  {i:2d}. {name}")

print(f"\nBefore vs After Comparison (first 10 examples):")
for i in range(10):
    if i < len(original_sample):
        orig = original_sample.iloc[i]
        clean = cleaned_sample.iloc[i]
        chars_saved = len(str(orig)) - len(str(clean))
        reduction_pct = (chars_saved / len(str(orig)) * 100) if len(str(orig)) > 0 else 0

        print(f"\n{i+1}. ORIGINAL: {orig}")
        print(f"   CLEANED:  {clean}")
        print(f"   SAVED:    {chars_saved} characters ({reduction_pct:.1f}% reduction)")

In [ ]:
"""
Statistical Analysis of Cleaning Results
----------------------------------------
Calculate quantitative metrics for cleaning effectiveness.
"""

print("Performing statistical analysis...")

# Calculate length statistics
original_lengths = original_names.str.len()
cleaned_lengths = df[PROTEIN_NAME_COLUMN].str.len()
chars_removed = original_lengths - cleaned_lengths

# Overall statistics
print("\nQuantitative Cleaning Analysis:")
print("-" * 50)
print(f"Total proteins processed: {len(df):,}")
print(f"\nLength Statistics:")
print(f"  Original average length: {original_lengths.mean():.1f} characters")
print(f"  Cleaned average length:  {cleaned_lengths.mean():.1f} characters")
print(f"  Average character reduction: {chars_removed.mean():.1f} characters")

# Modification statistics
names_modified = (chars_removed > 0).sum()
modification_rate = (names_modified / len(df)) * 100
print(f"\nModification Statistics:")
print(f"  Names modified: {names_modified:,}")
print(f"  Names unchanged: {(chars_removed == 0).sum():,}")
print(f"  Modification rate: {modification_rate:.1f}%")

# Compression statistics
original_total_chars = original_lengths.sum()
cleaned_total_chars = cleaned_lengths.sum()
total_chars_removed = original_total_chars - cleaned_total_chars
compression_ratio = (total_chars_removed / original_total_chars) * 100

print(f"\nCompression Statistics:")
print(f"  Total characters removed: {total_chars_removed:,}")
print(f"  Overall dataset compression: {compression_ratio:.2f}%")
print(f"  Maximum characters removed from single name: {chars_removed.max()}")

# Quality check
still_has_brackets = df[PROTEIN_NAME_COLUMN].str.contains(r'[\[\(]', na=False)
bracket_rate = (still_has_brackets.sum() / len(df)) * 100
print(f"\nQuality Check:")
print(f"  Names with remaining brackets: {still_has_brackets.sum():,} ({bracket_rate:.2f}%)")

In [ ]:
"""
Save Cleaned Dataset
-------------------
Export processed dataset in Parquet and CSV formats.
"""

print("Saving processed dataset...")

# Save as Parquet (recommended format)
parquet_path = os.path.join(OUTPUT_DIR, "cafa5_ppi_cleaned_protein_names.parquet")
df.to_parquet(parquet_path, index=False)
parquet_size = os.path.getsize(parquet_path) / (1024 * 1024)
print(f"\nParquet file saved:")
print(f"  Path: {parquet_path}")
print(f"  Size: {parquet_size:.1f} MB")

# Save as CSV backup
csv_path = os.path.join(OUTPUT_DIR, "cafa5_ppi_cleaned_protein_names.csv")
df.to_csv(csv_path, index=False)
csv_size = os.path.getsize(csv_path) / (1024 * 1024)
print(f"\nCSV backup saved:")
print(f"  Path: {csv_path}")
print(f"  Size: {csv_size:.1f} MB")



In [ ]:
"""
Generate Analysis Visualizations
--------------------------------
Create comprehensive visual analysis of cleaning results.
"""

print("Generating analysis visualizations...")

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'original': original_names,
    'cleaned': df[PROTEIN_NAME_COLUMN],
    'original_length': original_lengths,
    'cleaned_length': cleaned_lengths,
    'chars_removed': chars_removed
})

# Create figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Protein Name Cleaning Analysis Results', fontsize=16, fontweight='bold')

# Plot 1: Length comparison scatter
sample_size = min(5000, len(comparison_df))
viz_sample = comparison_df.sample(n=sample_size, random_state=42)

axes[0,0].scatter(viz_sample['original_length'], viz_sample['cleaned_length'],
                 alpha=0.6, s=1, c='blue')
max_len = viz_sample['original_length'].max()
axes[0,0].plot([0, max_len], [0, max_len], 'r--', alpha=0.7, label='No change')
axes[0,0].set_xlabel('Original Name Length (characters)')
axes[0,0].set_ylabel('Cleaned Name Length (characters)')
axes[0,0].set_title(f'Name Length Comparison\n(Sample of {sample_size:,} names)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Character reduction distribution
chars_removed_positive = chars_removed[chars_removed > 0]
if len(chars_removed_positive) > 0:
    axes[0,1].hist(chars_removed_positive, bins=50, alpha=0.7,
                   color='green', edgecolor='black')
axes[0,1].set_xlabel('Characters Removed per Name')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Distribution of Character Reduction')
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Length distribution comparison
length_bins = np.linspace(0, max(original_lengths.max(), cleaned_lengths.max()), 50)
axes[0,2].hist(original_lengths, bins=length_bins, alpha=0.7,
               label='Original', color='red', density=True)
axes[0,2].hist(cleaned_lengths, bins=length_bins, alpha=0.7,
               label='Cleaned', color='blue', density=True)
axes[0,2].set_xlabel('Name Length (characters)')
axes[0,2].set_ylabel('Density')
axes[0,2].set_title('Length Distribution Comparison')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# Plot 4: Modification overview pie chart
modified_count = names_modified
unmodified_count = (chars_removed == 0).sum()
axes[1,0].pie([modified_count, unmodified_count],
              labels=[f'Modified\n({modified_count:,})',
                     f'Unchanged\n({unmodified_count:,})'],
              autopct='%1.1f%%', colors=['lightcoral', 'lightblue'])
axes[1,0].set_title('Cleaning Impact Overview')

# Plot 5: Reduction by original length
original_length_bins = pd.cut(original_lengths, bins=10)
avg_reduction_by_bin = comparison_df.groupby(original_length_bins)['chars_removed'].mean()

bin_labels = [f"{int(interval.left)}-{int(interval.right)}"
              for interval in avg_reduction_by_bin.index]
axes[1,1].bar(range(len(bin_labels)), avg_reduction_by_bin.values,
              alpha=0.7, color='orange')
axes[1,1].set_xlabel('Original Name Length Bins')
axes[1,1].set_ylabel('Average Characters Removed')
axes[1,1].set_title('Character Reduction by Name Length')
axes[1,1].set_xticks(range(len(bin_labels)))
axes[1,1].set_xticklabels(bin_labels, rotation=45, ha='right')
axes[1,1].grid(True, alpha=0.3)

# Plot 6: Common removal patterns
removed_patterns = []
for idx in comparison_df[comparison_df['chars_removed'] > 0].index[:1000]:
    original = str(comparison_df.loc[idx, 'original'])

    if ', mitochondrial' in original:
        removed_patterns.append('Mitochondrial suffix')
    if '(EC ' in original:
        removed_patterns.append('EC numbers')
    if '[ubiquinone]' in original:
        removed_patterns.append('Ubiquinone cofactor')
    if '[NADP]' in original or '[NAD]' in original:
        removed_patterns.append('NAD/NADP cofactor')
    if '[Includes:' in original:
        removed_patterns.append('Complex descriptions')

if removed_patterns:
    pattern_counts = pd.Series(removed_patterns).value_counts().head(6)
    axes[1,2].barh(range(len(pattern_counts)), pattern_counts.values,
                   alpha=0.7, color='purple')
    axes[1,2].set_yticks(range(len(pattern_counts)))
    axes[1,2].set_yticklabels(pattern_counts.index)
    axes[1,2].set_xlabel('Frequency in Sample')
    axes[1,2].set_title('Common Removal Patterns\n(Sample Analysis)')
    axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
print("Visualization generation complete")
plt.show()

In [ ]:
"""
Detailed Examples by Cleaning Category
--------------------------------------
Show representative examples for each type of cleaning operation.
"""

print("Detailed before and after examples by cleaning category:")
print("=" * 70)

# Define example categories
example_categories = {
    'EC Number Removal': comparison_df[
        comparison_df['original'].str.contains(r'\(EC\s+[\d\.\-n]+\)', na=False)
    ].head(3),

    'Mitochondrial Suffix Removal': comparison_df[
        comparison_df['original'].str.contains(r',\s*mitochondrial', na=False, case=False)
    ].head(3),

    'Cofactor Bracket Removal': comparison_df[
        comparison_df['original'].str.contains(r'\[ubiquinone\]|\[NADP\]|\[NAD\]', na=False)
    ].head(3),

    'Complex Description Removal': comparison_df[
        comparison_df['original'].str.contains(r'\[Includes:', na=False)
    ].head(2),
}

for category, examples in example_categories.items():
    if len(examples) > 0:
        print(f"\n{category}:")
        print("-" * 70)
        for idx, (_, row) in enumerate(examples.iterrows(), 1):
            original = row['original']
            cleaned = row['cleaned']
            chars_saved = row['chars_removed']
            reduction_pct = (chars_saved / len(str(original)) * 100) if len(str(original)) > 0 else 0

            print(f"\nExample {idx}:")
            print(f"  BEFORE: {original}")
            print(f"  AFTER:  {cleaned}")
            print(f"  IMPACT: {chars_saved} characters removed ({reduction_pct:.1f}% reduction)")

print("\n" + "=" * 70)